In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("feature_dataset.csv")

# Features and label
X = df.drop(columns=['is_malicious'])
y = df['is_malicious']

# Split BEFORE any preprocessing to avoid leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import joblib  # For saving

# Define columns (adjust based on your dataset)
numerical_cols = ['response_time', 'html_content_length']  # Add all numerical features
categorical_cols = ['url']  # From notebook comment; add others if any

# Preprocessor: Scale nums, OneHot cats
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
    ],
    remainder='passthrough'  # Keep other columns (e.g., binary flags) as-is
)

# Fit on TRAIN only
preprocessor.fit(X_train)

# Transform train/test
X_train_prep = preprocessor.transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# Save preprocessor
joblib.dump(preprocessor, 'preprocessor.pkl')
print("✅ Preprocessor saved as preprocessor.pkl")

✅ Preprocessor saved as preprocessor.pkl


In [16]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_prep, y_train)

DecisionTreeClassifier(random_state=42)

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_prep, y_train)


RandomForestClassifier(random_state=42)

In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, name):
    y_pred = model.predict(X_test_prep)  # Use preprocessed test
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"\n{name} Results:")
    print(f"Accuracy: {acc:.2f}")
    print(f"Precision: {prec:.2f}")
    print(f"Recall: {rec:.2f}")
    print(f"F1-Score: {f1:.2f}")
    return [name, acc, prec, rec, f1]

results = []
results.append(evaluate_model(dt_model, "Decision Tree"))
results.append(evaluate_model(rf_model, "Random Forest"))


Decision Tree Results:
Accuracy: 0.40
Precision: 0.50
Recall: 0.67
F1-Score: 0.57

Random Forest Results:
Accuracy: 0.40
Precision: 0.50
Recall: 0.67
F1-Score: 0.57


In [21]:
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score"])
best_model_name = results_df.loc[results_df['F1-Score'].idxmax(), 'Model']
print("\nBest Model:", best_model_name)



Best Model: Decision Tree


In [23]:
import pickle
import os

# Save models
os.makedirs("projects/sql_injection/models", exist_ok=True)
pickle.dump(dt_model, open("decision_tree_model.pkl", "wb"))
pickle.dump(rf_model, open("random_forest_model.pkl", "wb"))

# Save best model
best_model = rf_model if best_model_name == "Random Forest" else dt_model
pickle.dump(best_model, open("best_model.pkl", "wb"))

# Save evaluation report
os.makedirs("projects/sql_injection/docs", exist_ok=True)
results_df.to_markdown("model_evaluation_report.md", index=False)
print("✅ Models and report saved successfully.")


✅ Models and report saved successfully.
